# Experimento Multi-LoRA: 4 Agentes Especializados sobre `Qwen/Qwen3-4B`

Este notebook entrena adaptadores QLoRA independientes para **Enrutador**, **Recuperador** y **Verificador**, y los combina en **un único modelo base NF4 4-bit (`Qwen/Qwen3-4B`)** junto con el **Redactor v2 de D4** (`Pablox0333/qwen3-4b-redactor-udec-d4`, revisión `739b031acbd5feef2c49f946b94722712209bce0`).

El objetivo es comparar en condiciones controladas:
1. **Control Base D4 (`--agentes ninguno`):** 4 agentes sin fine-tuning (`62/75`, `11/20`).
2. **Solo Redactor FT v2 (`--agentes redactor`):** arquitectura de Deliverable 4 (`60/75`, `10/20`).
3. **Multi-LoRA completo (`--agentes todos`):** los 4 agentes conmutando adaptadores por rol via `model.set_adapter(rol)`.

## 1. Entorno, GPU, repositorio (`exp/multi-lora-todos`) y dependencias fijadas

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
REPO_BRANCH = "exp/multi-lora-todos"
REPO_DIR = Path("/content/Proyecto_Git")

if not REPO_DIR.exists():
    res = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
        text=True,
    )
    if res.returncode != 0:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=False)

os.chdir(REPO_DIR)
print("Branch actual:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("Commit actual:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

subprocess.check_call(["nvidia-smi"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.23.0",
    "peft==0.21.3",
    "bitsandbytes==0.50.2",
    "accelerate==1.15.0",
    "huggingface-hub==1.33.0",
    "sentence-transformers==6.3.0",
    "matplotlib",
])
print("Dependencias instaladas correctamente.")

## 2. Persistencia opcional en Google Drive / Hugging Face y verificación antifuga

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/Proyecto_Git")
os.chdir(REPO_DIR)

# === Parámetros editables ===
USAR_GOOGLE_DRIVE = True          # Recomendado para no perder adaptadores/resultados si Colab desconecta
DRIVE_DIR = Path("/content/drive/MyDrive/UdeC_GenIA_Exp_MultiLoRA")
SUBIR_A_HF = False                # Cambia a True y define HF_USERNAME + token HF si quieres publicar los 3 LoRA
HF_USERNAME = "Pablox0333"
REENTRENAR = False                # Si False, reutiliza adaptadores ya guardados en local o en Drive
SKIP_SMOKE_TEST = False           # Pon True para ahorrar ~1-2 min de inferencia al final de cada entrenamiento

if USAR_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        for sub in ("entrenamiento", "evaluacion"):
            src = DRIVE_DIR / sub
            dst = REPO_DIR / "exp_multi_lora" / sub
            if src.exists():
                shutil.copytree(src, dst, dirs_exist_ok=True)
                print(f"Restaurado desde Drive: {src} -> {dst}")
    except Exception as exc:
        print("Drive no disponible fuera de Colab o rechazado:", exc)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if SUBIR_A_HF and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass("Token HF (write): ")
    os.environ["HF_TOKEN"] = HF_TOKEN

# Verificar determinismo antifuga y pruebas unitarias base (sin descargar modelo aún)
subprocess.check_call([sys.executable, "exp_multi_lora/scripts/construir_datasets_agentes.py"])
resumen = json.loads(Path("exp_multi_lora/datos/resumen_datasets_multi_lora.json").read_text(encoding="utf-8"))
for ag, info in resumen["agentes"].items():
    af = info["antifuga"]
    assert af["unidades_bloqueadas_solapadas"] == 0
    assert af["menciones_qid_eval"] == 0
    assert af["solapamiento_train_val"] == 0
    print(f"Dataset {ag:11s}: {info['train']:3d} train / {info['val']:2d} val | 0 fuga")

## 3. Entrenamiento secuencial QLoRA de los 3 agentes nuevos

Entrenamos en subprocesos separados para liberar toda la VRAM entre un agente y el siguiente:
- `enrutador` (210 train / 36 val, 105 pasos)
- `recuperador` (230 train / 39 val, 117 pasos)
- `verificador` (305 train / 68 val, 153 pasos)

El **Redactor** se mantiene exactamente en el adaptador oficial D4 v2 (`739b031`) para aislar el aporte neto de los otros tres agentes.

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from IPython.display import Image, display
import torch

os.chdir("/content/Proyecto_Git")

def _respaldar_a_drive(subcarpeta: str) -> None:
    if USAR_GOOGLE_DRIVE and DRIVE_DIR.exists():
        src = Path("exp_multi_lora") / subcarpeta
        if src.exists():
            shutil.copytree(src, DRIVE_DIR / subcarpeta, dirs_exist_ok=True)

AGENTES_A_ENTRENAR = ["enrutador", "recuperador", "verificador"]

for ag in AGENTES_A_ENTRENAR:
    adapter_dir = Path("exp_multi_lora/entrenamiento") / ag / "adapter"
    if (adapter_dir / "adapter_model.safetensors").is_file() and not REENTRENAR:
        print(f"[{ag}] Ya existe adaptador entrenado en {adapter_dir}; saltando (REENTRENAR=False).")
        continue

    cmd = [sys.executable, "exp_multi_lora/scripts/entrenar_agente.py", "--agente", ag]
    if SKIP_SMOKE_TEST:
        cmd.append("--skip-smoke-test")
    if SUBIR_A_HF and HF_USERNAME:
        cmd.extend(["--hf-repo", f"{HF_USERNAME}/qwen3-4b-{ag}-udec-exp"])

    print(f"\n==================== Entrenando {ag.upper()} ====================", flush=True)
    subprocess.check_call(cmd)
    _respaldar_a_drive("entrenamiento")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n--- Resumen de entrenamientos ---")
for ag in AGENTES_A_ENTRENAR:
    m_path = Path("exp_multi_lora/entrenamiento") / ag / "training_metrics.json"
    if m_path.is_file():
        m = json.loads(m_path.read_text(encoding="utf-8"))
        r = m["resultados"]
        print(
            f"{ag:12s} | {r['tiempo_entrenamiento_min']:5.2f} min | "
            f"eval_loss: {r['eval_loss_inicial']} -> {r['mejor_eval_loss']} (step {r['mejor_step']}) | "
            f"truncados train={m['datos']['tokens_train']['truncados']}"
        )
        png = Path("exp_multi_lora/entrenamiento") / ag / "loss_curve.png"
        if png.is_file():
            display(Image(filename=str(png), width=620))

## 4. Carga Multi-LoRA en un solo `Qwen/Qwen3-4B` y prueba de aislamiento cruzado

Comprobamos que los 4 adaptadores (`enrutador`, `recuperador`, `redactor`, `verificador`) conviven sobre la misma base NF4 y producen logits distintos frente al modelo base y distintos entre sí.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.chdir("/content/Proyecto_Git")
sys.path.insert(0, str(Path("Deliverable 4/scripts").resolve()))
sys.path.insert(0, str(Path("Deliverable 3/Deliverable3/agentes").resolve()))
sys.path.insert(0, str(Path("exp_multi_lora/scripts").resolve()))

import evaluar_d4 as ed4
import evaluar_multi_lora as eml

ROLES_ACTIVOS = ("enrutador", "recuperador", "redactor", "verificador")
fuentes = eml.resolver_fuentes_adaptadores_default()

corpus = ed4.construir_corpus_d3(use_e5=True)
model, tokenizer, compute_dtype_str, base_commit = eml.cargar_modelo_y_multi_lora(
    eml.MODELO_BASE_DEFAULT,
    fuentes,
    ROLES_ACTIVOS,
)
aislamiento = eml.verificar_aislamiento_multi_lora(model, tokenizer, ROLES_ACTIVOS)
print("Compute dtype:", compute_dtype_str)
print("Aislamiento Multi-LoRA:")
print(json.dumps(aislamiento, ensure_ascii=False, indent=2))

## 5. Comparativa pareada en vivo (`Solo Redactor D4` vs `Multi-LoRA 4 agentes`)

Evaluamos casos diagnósticos donde el Redactor v2 o el sistema D4 tuvieron fricción con Enrutador / Recuperador / Verificador base (`H06`, `H07`, `H22`, `N05`, `N12`). Por defecto usamos la misma decodificación controlada de D4 (`greedy=False`, `T=0.7, top_p=0.8, top_k=20`, semilla derivada); cambia `GREEDY = True` si quieres decodificación voraz determinista.

In [ ]:
GREEDY = False
EVAL_OUT_DIR = Path("exp_multi_lora/evaluacion")
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

gen_solo_redactor = eml.MultiLoraGenerator(
    model,
    tokenizer,
    adaptadores_activos={"redactor": "redactor"},
    greedy=GREEDY,
    verbose=True,
)
gen_multi_lora = eml.MultiLoraGenerator(
    model,
    tokenizer,
    adaptadores_activos={r: r for r in ROLES_ACTIVOS},
    greedy=GREEDY,
    verbose=True,
)

# 1) Casos difíciles de D3
preg_d3_diag = ed4.cargar_preguntas_evaluacion(filtro_qids={"dificil_25:H06", "dificil_25:H07", "dificil_25:H22"})
comp_d3 = eml.ejecutar_comparativa_pareada(
    preg_d3_diag,
    corpus,
    gen_solo_redactor,
    gen_multi_lora,
    "solo_redactor_d4",
    "multi_lora_4",
    EVAL_OUT_DIR / "comparativa_diagnostico_d3.json",
)

# 2) Casos diagnósticos de nuevas_20
preg_n20_diag = ed4.cargar_preguntas_desde_csv(
    Path("Deliverable 4/datos/preguntas_nuevas_20.csv"),
    nombre_set="nuevas_20",
    filtro_qids={"nuevas_20:N05", "nuevas_20:N12"},
)
comp_n20 = eml.ejecutar_comparativa_pareada(
    preg_n20_diag,
    corpus,
    gen_solo_redactor,
    gen_multi_lora,
    "solo_redactor_d4",
    "multi_lora_4",
    EVAL_OUT_DIR / "comparativa_diagnostico_n20.json",
)

for c in comp_d3 + comp_n20:
    a, b = c["solo_redactor_d4"], c["multi_lora_4"]
    print(f"\n=== {c['qid']} ({c['categoria']}) ===")
    print(f"Gold ({c['gold_fuente']}): {c['gold_dato']}")
    print(f"[Solo Redactor] ciclos={a['ciclos']} | aprobado={a['veredicto'].get('aprobado')} | {a['respuesta'][:240]}")
    print(f"[Multi-LoRA   ] ciclos={b['ciclos']} | aprobado={b['veredicto'].get('aprobado')} | {b['respuesta'][:240]}")

_respaldar_a_drive("evaluacion")

## 6. Benchmark completo: `nuevas_20` (20 preguntas ciegas) y D3 (75 preguntas)

Ejecuta `nuevas_20` primero (~3–6 min en A100 / ~12–18 min en T4) y luego las 75 completas de D3. Ambos soportan reanudación automática por pregunta.

In [ ]:
import torch
import resumir_comparativa as rc

prompts_sha = ed4.verificar_prompts_d4()
meta_multi = {
    "modelo_base": eml.MODELO_BASE_DEFAULT,
    "revision_modelo_base": base_commit,
    "roles_con_lora": list(ROLES_ACTIVOS),
    "fuentes_por_rol": {r: {"fuente": fuentes[r][0], "revision": fuentes[r][1]} for r in ROLES_ACTIVOS},
    "hardware": {
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "compute_dtype": compute_dtype_str,
    },
    "greedy": GREEDY,
    "aislamiento_multi_lora": aislamiento,
    "prompts_sha256": prompts_sha,
    "versiones": ed4._resolver_versiones_eval(),
}

# --- A) nuevas_20 ---
preg_n20 = ed4.cargar_preguntas_desde_csv(
    Path("Deliverable 4/datos/preguntas_nuevas_20.csv"),
    nombre_set="nuevas_20",
)
gen_n20 = eml.MultiLoraGenerator(
    model, tokenizer, adaptadores_activos={r: r for r in ROLES_ACTIVOS}, greedy=GREEDY
)
ed4.ejecutar_evaluacion_lote(
    preg_n20,
    corpus,
    gen_n20,
    output_dir=EVAL_OUT_DIR,
    condicion="multi_lora:4_agentes",
    metadata_extra=meta_multi,
    output_prefix="multi_lora_nuevas20",
)
_respaldar_a_drive("evaluacion")

res_n20 = [
    rc.resumir_corrida(
        "Control Base D4",
        Path("Deliverable 4/evaluacion/resultados_nuevas20_base_D4_auditados.csv"),
        Path("Deliverable 4/evaluacion/telemetria_nuevas20_base_D4.json"),
    ),
    rc.resumir_corrida(
        "Solo Redactor D4",
        Path("Deliverable 4/evaluacion/resultados_nuevas20_ft_D4_auditados.csv"),
        Path("Deliverable 4/evaluacion/telemetria_nuevas20_ft_D4.json"),
    ),
    rc.resumir_corrida(
        "Multi-LoRA (4 agentes)",
        EVAL_OUT_DIR / "resultados_multi_lora_nuevas20.csv",
        EVAL_OUT_DIR / "telemetria_multi_lora_nuevas20.json",
    ),
]
print("### Comparativa nuevas_20 ###")
print(rc.formatear_tabla_markdown(res_n20))
print("Desglose llamadas Multi-LoRA:", json.dumps(res_n20[-1]["telemetria"]["llamadas_por_adaptador"], indent=2))

In [ ]:
# --- B) Benchmark completo de 75 preguntas D3 ---
CORRER_75_COMPLETO = True  # Cambia a True cuando quieras correr las 75 preguntas completas

if CORRER_75_COMPLETO:
    preg_75 = ed4.cargar_preguntas_evaluacion()
    gen_75 = eml.MultiLoraGenerator(
        model, tokenizer, adaptadores_activos={r: r for r in ROLES_ACTIVOS}, greedy=GREEDY
    )
    ed4.ejecutar_evaluacion_lote(
        preg_75,
        corpus,
        gen_75,
        output_dir=EVAL_OUT_DIR,
        condicion="multi_lora:4_agentes",
        metadata_extra=meta_multi,
        output_prefix="multi_lora_75",
    )
    _respaldar_a_drive("evaluacion")

    res_75 = [
        rc.resumir_corrida(
            "Control Base D4",
            Path("Deliverable 4/evaluacion/resultados_control_base_D4_auditados.csv"),
            Path("Deliverable 4/evaluacion/telemetria_control_base_D4.json"),
        ),
        rc.resumir_corrida(
            "Solo Redactor D4",
            Path("Deliverable 4/evaluacion/resultados_agentes_D4_auditados.csv"),
            Path("Deliverable 4/evaluacion/telemetria_llamadas.json"),
        ),
        rc.resumir_corrida(
            "Multi-LoRA (4 agentes)",
            EVAL_OUT_DIR / "resultados_multi_lora_75.csv",
            EVAL_OUT_DIR / "telemetria_multi_lora_75.json",
        ),
    ]
    print("### Comparativa 75 preguntas D3 ###")
    print(rc.formatear_tabla_markdown(res_75))
    diffs_75 = rc.diferencias_por_pregunta(
        Path("Deliverable 4/evaluacion/resultados_agentes_D4_auditados.csv"),
        EVAL_OUT_DIR / "resultados_multi_lora_75.csv",
    )
    print(f"Preguntas con diferencias frente a Solo Redactor D4: {len(diffs_75)}")

## 7. Ablación opcional: solo `Redactor + Verificador` (`--agentes redactor,verificador`)

Si quieres medir por separado cuánto aporta fine-tunear únicamente el **Verificador** (dejando Enrutador y Recuperador en base), habilita la celda siguiente.

In [ ]:
CORRER_ABLACION_RED_VER = False

if CORRER_ABLACION_RED_VER:
    gen_rv = eml.MultiLoraGenerator(
        model,
        tokenizer,
        adaptadores_activos={"redactor": "redactor", "verificador": "verificador"},
        greedy=GREEDY,
    )
    ed4.ejecutar_evaluacion_lote(
        preg_n20,
        corpus,
        gen_rv,
        output_dir=EVAL_OUT_DIR,
        condicion="multi_lora:redactor+verificador",
        metadata_extra={**meta_multi, "roles_con_lora": ["redactor", "verificador"]},
        output_prefix="ablacion_red_ver_nuevas20",
    )
    _respaldar_a_drive("evaluacion")
    print("Ablación redactor+verificador en nuevas_20 completada.")